# Exercises: K-Means

**Practical Machine Learning** · Sayan CHAKI · LIRIS, Université Lyon 2

Three exercises:

| # | Topic | Main skills |
|---|---|---|
| 1 | Clustering **iris** without the labels | choosing k (elbow, silhouette), ARI, crosstabs, is scaling always right? |
| 2 | **Colour quantisation** of a photograph | K-means as compression, the distortion/size trade-off, initialisation |
| 3 | **When K-means fails** | anisotropic and non-convex clusters, Gaussian mixtures |

**Open in Colab.** In [colab.research.google.com](https://colab.research.google.com) choose *File → Upload notebook* (or open it from Google Drive). Everything uses datasets shipped with scikit-learn, so no download or upload of data is needed.

**Rules of the game**
* Keep all `random_state` / seeds as given, so results are comparable across the class.
* Never use the test set to choose a model or a hyperparameter; it is opened **once**, at the end.
* Cells marked `# TODO` are yours. Cells marked `# CHECK` test your work: run them, they must pass.
* Questions marked ✍️ need a short written answer (2 to 4 sentences, with numbers from your results).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, load_sample_image, make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

---
# Exercise 1 · Clustering iris without the labels

**Context.** 150 flowers, 4 measurements (sepal and petal length and width, in cm). We pretend the species are unknown, cluster the flowers, and only at the end compare the clusters with the species.

### 1.1 How many clusters?
Standardise the four features into `Xs`. For `k` from 2 to 8, fit `KMeans(n_clusters=k, n_init=10, random_state=0)` on `Xs` and store the inertia in `inertias` and the silhouette score in `silhouettes`. Plot both curves side by side.

In [ ]:
iris = load_iris()
X = iris.data
species = iris.target        # used ONLY for evaluation at the end
print(pd.DataFrame(X, columns=iris.feature_names).describe().round(2))

In [ ]:
# TODO
Xs = None
ks = range(2, 9)
inertias, silhouettes = [], []

In [ ]:
# CHECK (run this cell, it must pass)
assert Xs is not None and np.allclose(Xs.mean(axis=0), 0) and np.allclose(Xs.std(axis=0), 1)
assert len(inertias) == 7 and all(np.diff(inertias) < 0), "inertia must decrease with k"
assert len(silhouettes) == 7
print("✅ model-selection curves computed")

In [ ]:
# A look at the data with the true species (for discussion only)
fig, ax = plt.subplots()
for c, name in enumerate(iris.target_names):
    ax.scatter(X[species == c, 2], X[species == c, 3], label=name, s=20)
ax.set_xlabel("petal length (cm)"); ax.set_ylabel("petal width (cm)"); ax.legend(); plt.show()

### ✍️ Question 1
There are three species, yet which `k` does the silhouette prefer? Use the plot above to explain why. What does this say about using the silhouette (or the elbow) as the sole criterion for choosing `k`?

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*

### 1.2 Compare with the species
Fit `KMeans(n_clusters=3, n_init=10, random_state=0)` twice: on the standardised data `Xs` (labels `lab_scaled`) and on the raw data `X` (labels `lab_raw`). Compute the adjusted Rand index of each with `species` (`ari_scaled`, `ari_raw`) and show `pd.crosstab(species, labels)` for both.

In [ ]:
# TODO
lab_scaled = lab_raw = None
ari_scaled = ari_raw = None

In [ ]:
# CHECK (run this cell, it must pass)
assert 0 < ari_scaled < 1 and 0 < ari_raw < 1
assert sorted(np.unique(lab_scaled)) == [0, 1, 2]
print("✅ ARI computed")

### ✍️ Question 2
Here, clustering the **raw** data matches the species better than clustering the standardised data. Using the standard deviations printed above, explain why. Is "always standardise before K-means" a law?

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*

---
# Exercise 2 · Colour quantisation

**Context.** A photograph is a set of pixels, each a point in the 3-D colour space (R, G, B). K-means on those points finds a palette of `k` colours; replacing each pixel by its cluster centre compresses the image.

### 2.1 The image

In [ ]:
img = load_sample_image("china.jpg") / 255.0          # (427, 640, 3), values in [0, 1]
pixels = img.reshape(-1, 3)
print("image shape:", img.shape, "  pixels:", len(pixels))
print("distinct colours:", len(np.unique((pixels * 255).astype(np.uint8), axis=0)))
plt.imshow(img); plt.axis("off"); plt.show()

### 2.2 Quantise
Write `quantize(img, k, n_sample=10_000, seed=0)`:
1. flatten the image to `(n_pixels, 3)`;
2. fit `KMeans(n_clusters=k, n_init=4, random_state=seed)` on a random sample of `n_sample` pixels (`np.random.default_rng(seed).choice(...)`), because fitting on all 273 280 pixels is slow;
3. predict the cluster of **every** pixel and replace it by its centre;
4. return the quantised image with the original shape, and the fitted model.

In [ ]:
# TODO
def quantize(img, k, n_sample=10_000, seed=0):
    raise NotImplementedError("TODO")

In [ ]:
# CHECK (run this cell, it must pass)
q8, km8 = quantize(img, 8)
assert q8.shape == img.shape
assert len(np.unique(q8.reshape(-1, 3), axis=0)) <= 8
assert q8.min() >= 0 and q8.max() <= 1
print("✅ quantize works")

### 2.3 Distortion versus size
For `k` in `[2, 4, 8, 16, 32, 64]` compute
* the mean squared error between `img` and the quantised image → list `mses`;
* the compression ratio `24 * n / (n * ceil(log2 k) + 24 * k)` where `n` is the number of pixels (original: 24 bits per pixel; quantised: an index per pixel plus a palette of `k` colours of 24 bits) → list `ratios`.

Show the six images in a grid, then plot MSE against `k`.

In [ ]:
# TODO
k_list = [2, 4, 8, 16, 32, 64]
mses, ratios = [], []

In [ ]:
# CHECK (run this cell, it must pass)
assert len(mses) == 6 and all(np.diff(mses) < 0), "MSE should decrease as k grows"
assert abs(ratios[0] - 24) < 0.1, "k=2 needs 1 bit per pixel: ratio close to 24"
print("✅ trade-off computed")

### ✍️ Question 3
Which palette size would you pick for this photo and why? Relate MSE to the K-means objective: what exactly does K-means minimise here?

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*

### 2.4 Does initialisation matter?
On a fixed sample of 10 000 pixels (`seed 0`), fit `KMeans(n_clusters=16, n_init=1)` with `init="random"` and with `init="k-means++"`, each for `random_state` 0 to 9. Store the inertias in two lists `inert_random`, `inert_pp` and compare mean, min and max.

In [ ]:
# TODO
sample = pixels[np.random.default_rng(0).choice(len(pixels), 10_000, replace=False)]
inert_random, inert_pp = [], []

In [ ]:
# CHECK (run this cell, it must pass)
assert len(inert_random) == 10 and len(inert_pp) == 10
print("✅ initialisation comparison done")

### ✍️ Question 4
Summarise the comparison. Why do several runs with different seeds end at different inertias, and what does `n_init` do about it?

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*

---
# Exercise 3 · When K-means fails

K-means assigns each point to the nearest centre, so each cluster is a convex region (a Voronoi cell), and the objective favours round clusters of similar spread.

### 3.1 Stretched clusters
Run the cell to create three elongated clusters. Then fit `KMeans(3, n_init=10, random_state=0)` and `GaussianMixture(3, covariance_type="full", random_state=0)`, store their ARIs with the true labels in `ari_km` and `ari_gmm`, and plot both clusterings side by side.

In [ ]:
Xa, ya = make_blobs(n_samples=600, centers=3, random_state=170)
Xa = Xa @ np.array([[0.6, -0.64], [-0.41, 0.85]])      # linear stretch
plt.scatter(*Xa.T, c=ya, s=10); plt.title("true groups"); plt.show()

In [ ]:
# TODO
ari_km = ari_gmm = None

In [ ]:
# CHECK (run this cell, it must pass)
assert ari_gmm > ari_km + 0.2
print(f"✅ K-means {ari_km:.3f} vs GMM {ari_gmm:.3f}")

### 3.2 Non-convex clusters
Fit `KMeans(2, n_init=10, random_state=0)` on the two moons below and store the ARI in `ari_moons`. Plot the result.

In [ ]:
Xm, ym = make_moons(n_samples=400, noise=0.05, random_state=0)

In [ ]:
# TODO
ari_moons = None

In [ ]:
# CHECK (run this cell, it must pass)
assert ari_moons < 0.6
print("✅ done")

### ✍️ Question 5
Explain each failure in terms of the geometry of K-means. Why does the Gaussian mixture fix 3.1 but would not fix 3.2? Name a method suited to 3.2.

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*